# Stepping through zero crossings

A small signal that oscillates around a centre that is not zero. Crossings of the centre come twice per period, so the spacing between them gives the period. This walks through the three array steps from the Features from data notebook, centring, taking the sign, and finding where the sign changes, then steps through the crossings one at a time.

In [1]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

## The data

About two and a half cycles sampled every 0.05 s. The signal was made with a period of 0.55 s, so the answer can be checked.

In [2]:
t = np.round(0.05 * np.arange(26), 2)
a = np.array([0.51, 0.68, 0.81, 0.76, 0.50, 0.24, 0.00, -0.19, -0.15, -0.06, 0.22, 0.49, 0.68,
              0.82, 0.74, 0.50, 0.27, -0.02, -0.19, -0.16, -0.07, 0.22, 0.51, 0.69, 0.80, 0.74])

## Finding the crossings

Subtract the mean so the signal oscillates around zero. `np.sign` turns each sample into +1 or −1, and `np.diff` of the signs is zero between two samples on the same side and ±2 between two samples on opposite sides. Index `k` in `crossing` means the sign changes between sample `k` and sample `k + 1`.

In [3]:
a_centered = a - a.mean()
sign = np.sign(a_centered)
change = np.diff(sign)
crossing = np.where(change != 0)[0]

print("mean:    ", round(a.mean(), 3))
print("sign:    ", sign.astype(int))
print("diff:    ", change.astype(int))
print("crossing:", crossing)

mean:     0.359
sign:     [ 1  1  1  1  1 -1 -1 -1 -1 -1 -1  1  1  1  1  1 -1 -1 -1 -1 -1 -1  1  1
  1  1]
diff:     [ 0  0  0  0 -2  0  0  0  0  0  2  0  0  0  0 -2  0  0  0  0  0  2  0  0
  0]
crossing: [ 4 10 15 21]


## Step through it

Run the cell, then use the buttons. **Next** and **Back** move one stage, **Reset** returns to the raw data. The first stages build the arrays from the cell above; after that, each click marks one crossing. The black ring marks the sample before the sign change, and the arrow is the time back to the previous crossing.

In [4]:
INTRO = [
    "the signal oscillates, but not around zero",
    f"its centre is the mean, {a.mean():.2f}",
    "subtract the mean: a_centered = a - a.mean() oscillates around zero",
    "np.sign(a_centered): +1 above zero, -1 below",
    f"np.diff(sign) != 0 where neighbours have opposite signs: crossing = {crossing.tolist()}",
]
n_stages = len(INTRO) + len(crossing) + 1          # intro, one per crossing, result
above, below = sign > 0, sign < 0

with plt.ioff():
    fig, ax = plt.subplots(figsize=(9, 4.2))
fig.canvas.header_visible = False


def draw(stage):
    """Draw the figure for one stage."""
    ax.cla()
    ax.set_xlabel("time (s)")
    if stage < 2:
        ax.plot(t, a, "-o", ms=4, lw=1, color="C0", zorder=1)
        ax.axhline(0, color="gray", lw=0.5)
        ax.set_ylim(-0.6, 1.05)
        ax.set_ylabel("acceleration (g)")
        if stage == 0:
            for j in range(len(a)):
                ax.annotate(str(j), (t[j], a[j]), textcoords="offset points", xytext=(0, 7),
                            ha="center", fontsize=7, color="gray")
        else:
            ax.axhline(a.mean(), color="C1", linestyle="--", label="mean")
        message = INTRO[stage]
    else:
        ax.plot(t, a_centered, "-", lw=1, color="C0", zorder=1)
        ax.axhline(0, color="gray", linestyle=":", label="zero")
        ax.set_ylim(-0.75, 0.75)
        ax.set_ylabel("centred acceleration (g)")
        if stage == 2:
            ax.plot(t, a_centered, "o", ms=4, color="C0")
        else:
            ax.plot(t[above], a_centered[above], "o", ms=6, color="C0", label="sign +1")
            ax.plot(t[below], a_centered[below], "o", ms=6, color="C3", label="sign -1")
        if stage >= 4:
            for j, k in enumerate(crossing):
                ax.plot(t[k:k + 2], a_centered[k:k + 2], "-", lw=4, color="C1", alpha=0.6,
                        label="sign change" if j == 0 else None)
                ax.annotate(str(k), (t[k], a_centered[k]), textcoords="offset points",
                            xytext=(-9, 0), ha="right", va="center", fontsize=8, color="C1")
        if stage < len(INTRO):
            message = INTRO[stage]
        else:
            n_done = min(stage - len(INTRO) + 1, len(crossing))
            done = crossing[:n_done]
            ax.plot(t[done], np.zeros(n_done), "D", ms=7, color="C2", label="crossings found")
            y = 0.62
            for j in range(1, n_done):
                k0, k1 = done[j - 1], done[j]
                ax.annotate("", xy=(t[k1], y), xytext=(t[k0], y),
                            arrowprops=dict(arrowstyle="<->", color="k"))
                ax.text((t[k0] + t[k1]) / 2, y + 0.02, f"{t[k1] - t[k0]:.2f} s",
                        ha="center", va="bottom", fontsize=8)
            spacing = np.diff(t[done])
            if stage < n_stages - 1:
                k = done[-1]
                ax.plot([t[k]], [a_centered[k]], "o", ms=20, mfc="none", mec="k", mew=2)
                message = (f"index {k}: sign {int(sign[k]):+d} to {int(sign[k + 1]):+d} "
                           f"between samples {k} and {k + 1}, crossing at t[{k}] = {t[k]:.2f} s")
                if n_done > 1:
                    message += f"\n{spacing[-1]:.2f} s after the previous crossing"
            else:
                T = 2 * spacing.mean()
                message = (f"spacings {spacing.round(2).tolist()} s are half periods; "
                           f"T = 2 × mean = {T:.3f} s")
    ax.legend(loc="lower right", fontsize=8)
    ax.set_title(f"{stage + 1}/{n_stages}   {message}", fontsize=10)
    fig.canvas.draw_idle()


stage = 0
back = widgets.Button(description="Back", icon="arrow-left")
step = widgets.Button(description="Next", icon="arrow-right", button_style="primary")
reset = widgets.Button(description="Reset", icon="refresh")


def go(new_stage):
    global stage
    stage = max(0, min(new_stage, n_stages - 1))
    back.disabled = stage == 0
    step.disabled = stage == n_stages - 1
    draw(stage)


back.on_click(lambda _: go(stage - 1))
step.on_click(lambda _: go(stage + 1))
reset.on_click(lambda _: go(0))
go(0)
display(widgets.VBox([widgets.HBox([reset, back, step]), fig.canvas]))

/var/folders/q0/kxmqm5c95n7cxc6mmqklw1k40000gq/T/ipykernel_66235/2398276189.py:73: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend(loc="lower right", fontsize=8)


The signal was made with a period of 0.55 s, so the true half period is 0.275 s. Each spacing is 0.30 s or 0.25 s because `t[k]` is the sample *before* the crossing, which can be up to one sample (0.05 s) early. The errors partly cancel, so the mean of the spacings is closer than any single spacing, and more cycles help further.